In [34]:
import pandas as pd
import numpy as np

In [2]:
# read payment csv files 
df_payment = pd.read_csv("payment_report.csv")
print(df_payment)

    report_month payment_group  product_id  source_id       volume
0        2023-01       payment          12         45    624110375
1        2023-01       payment          17         45    335715113
2        2023-01       payment          18         45    737784466
3        2023-01       payment          19         45    120963069
4        2023-01       payment          20         45    319653158
..           ...           ...         ...        ...          ...
914      2023-04       payment       15067         45      1504000
915      2023-04        refund        1976         37   3542271587
916      2023-04        refund        1976         38  13831708189
917      2023-04        refund        1976         39   1905435543
918      2023-04        refund        1976         39   3679922071

[919 rows x 5 columns]


In [3]:
# read product csv files 
df_product = pd.read_csv("product.csv")
print(df_product)


     product_id category team_own
0            17  PXXXXXB      ASD
1            18  PXXXXXB      ASD
2            20  PXXXXXB      ASD
3           287  PXXXXXB      ASD
4           372  PXXXXXB      ASD
..          ...      ...      ...
487         321  PXXXXXV      ASD
488         322  PXXXXXV      ASD
489         341  PXXXXXV      ASD
490         342  PXXXXXV      ASD
491         367  PXXXXXV      ASD

[492 rows x 3 columns]


In [4]:
# read transactions csv file
df_transactions = pd.read_csv("transactions.csv")
print(df_transactions.head(5))

   transaction_id  merchant_id   volume  transType  transStatus   sender_id  \
0      3002692434            5   100000         24            1  10199794.0   
1      3002692437          305    20000          2            1  14022211.0   
2      3001960110         7255    48605         22            1         NaN   
3      3002680710         2270  1500000          2            1  10059206.0   
4      3002680713         2275    90000          2            1  10004711.0   

   receiver_id extra_info      timeStamp  
0     199794.0        NaN  1682932054455  
1   14022211.0        NaN  1682932054912  
2   10530940.0        NaN  1682932055000  
3      59206.0        NaN  1682932055622  
4       4711.0        NaN  1682932056197  


In [5]:
# merge payment and product 
payment_product = df_payment.merge( df_product, on="product_id", how = "left" )
print(payment_product)

    report_month payment_group  product_id  source_id       volume  category  \
0        2023-01       payment          12         45    624110375   PXXXXXT   
1        2023-01       payment          17         45    335715113   PXXXXXB   
2        2023-01       payment          18         45    737784466   PXXXXXB   
3        2023-01       payment          19         45    120963069  PXXXXXM2   
4        2023-01       payment          20         45    319653158   PXXXXXB   
..           ...           ...         ...        ...          ...       ...   
914      2023-04       payment       15067         45      1504000   PXXXXXR   
915      2023-04        refund        1976         37   3542271587       NaN   
916      2023-04        refund        1976         38  13831708189       NaN   
917      2023-04        refund        1976         39   1905435543       NaN   
918      2023-04        refund        1976         39   3679922071       NaN   

    team_own  
0        ASD  
1        

Part I: EDA - Explore Data Analysis


In [6]:
# checking after merging
print(df_payment.shape)
print(payment_product.shape)
print(df_transactions.shape)

(919, 5)
(919, 7)
(1324002, 9)


In [7]:
# cheking missing value
payment_product.isna().sum()


report_month      0
payment_group     0
product_id        0
source_id         0
volume            0
category         22
team_own         22
dtype: int64

In [8]:
df_transactions.isna().sum()


transaction_id          0
merchant_id             0
volume                  0
transType               0
transStatus             0
sender_id           49059
receiver_id        164795
extra_info        1317907
timeStamp               0
dtype: int64

In [9]:
#drop missing value in transactions
df_transactions = df_transactions.dropna(
    subset=["sender_id", "receiver_id"],
    how="all"
)


In [10]:
# checking duplicated payment
key_cols = ["report_month", "payment_group", "product_id", "source_id"]

duplicates = payment_product.duplicated(subset=key_cols).sum()

print("Number of duplicates:", duplicates)

Number of duplicates: 5


In [11]:
# Dropping 5 duplicates
after_dr_payment_product = payment_product.drop_duplicates(subset= key_cols)


In [12]:
# checking duplicated transactions
print("Duplicated transaction_id:", df_transactions["transaction_id"].duplicated().sum())

Duplicated transaction_id: 28


In [13]:
# dropping 28 duplicates
df_transactions = df_transactions.drop_duplicates(subset=["transaction_id"])



In [14]:
df_transactions["transaction_id"].is_unique

True

In [15]:
# checking incorrect data types
payment_product.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
 5   category       897 non-null    object
 6   team_own       897 non-null    object
dtypes: int64(3), object(4)
memory usage: 50.4+ KB


In [16]:
# Convert data type of 'report_month' to datetime
payment_product["report_month"] = pd.to_datetime(payment_product["report_month"])

In [17]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1321387 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1321387 non-null  int64  
 1   merchant_id     1321387 non-null  int64  
 2   volume          1321387 non-null  int64  
 3   transType       1321387 non-null  int64  
 4   transStatus     1321387 non-null  int64  
 5   sender_id       1274916 non-null  float64
 6   receiver_id     1159182 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1321387 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 100.8+ MB


In [18]:
# Convert data type of 'sender_id' and 'receiver_id' to int64
df_transactions["sender_id"] = df_transactions["sender_id"].astype("Int64")
df_transactions["receiver_id"] = df_transactions["receiver_id"].astype("Int64")

In [19]:
# Summary statistics
print(payment_product.describe())

                        report_month    product_id   source_id        volume
count                            919    919.000000  919.000000  9.190000e+02
mean   2023-02-19 06:05:05.549510400   1192.517954   44.875952  1.978574e+08
min              2023-01-01 00:00:00      3.000000   37.000000  5.500000e+03
25%              2023-02-01 00:00:00    640.000000   45.000000  1.250000e+06
50%              2023-03-01 00:00:00   1059.000000   45.000000  7.982015e+06
75%              2023-04-01 00:00:00   1585.000000   45.000000  5.447599e+07
max              2023-04-01 00:00:00  15067.000000   45.000000  1.383171e+10
std                              NaN   1293.463329    0.910995  8.367595e+08


In [20]:
print(df_transactions.describe())

       transaction_id   merchant_id        volume     transType   transStatus  \
count    1.321387e+06  1.321387e+06  1.321387e+06  1.321387e+06  1.321387e+06   
mean     3.002233e+09  2.462103e+03  2.382381e+05  6.934112e+00 -1.193622e+01   
std      1.043623e+07  3.306412e+03  9.644103e+05  7.397025e+00  5.558634e+01   
min      3.000000e+09  5.000000e+00  1.000000e+00  2.000000e+00 -1.333000e+03   
25%      3.001121e+09  3.050000e+02  1.000000e+04  2.000000e+00  1.000000e+00   
50%      3.002200e+09  2.250000e+03  3.000000e+04  2.000000e+00  1.000000e+00   
75%      3.003255e+09  2.270000e+03  1.000000e+05  8.000000e+00  1.000000e+00   
max      6.000066e+09  1.625250e+05  7.869148e+07  3.000000e+01  1.000000e+00   

              sender_id       receiver_id     timeStamp  
count         1274916.0         1159182.0  1.321387e+06  
mean   103392211.603019  208488236.021155  1.683130e+12  
std    623425527.063002  928782507.772745  1.707796e+08  
min          10000007.0             -6

In [21]:
# check incorrect values in 'payment_product'
print(payment_product[payment_product["volume"] <= 0])



Empty DataFrame
Columns: [report_month, payment_group, product_id, source_id, volume, category, team_own]
Index: []


In [22]:
# check incorrect values in 'transactions'
print(df_transactions[df_transactions["volume"] <= 0])


Empty DataFrame
Columns: [transaction_id, merchant_id, volume, transType, transStatus, sender_id, receiver_id, extra_info, timeStamp]
Index: []


Part 2: Data Wrangling

In [23]:
# Top 3 product_ids with the highest volume.
top_products = ( 
    payment_product
    .groupby("product_id")["volume"]
    .sum()
    .reset_index()
)
top_products = top_products.sort_values(
    by = "volume", ascending= False
)

print(top_products.head(3))

     product_id       volume
279        1976  61797583647
43          429  14667676567
38          372  13713658515


In [24]:
# Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule
abnormal_products = (
    payment_product
    .groupby("product_id")["team_own"]
    .nunique()
    .reset_index(name="num_teams")
)

abnormal_products = abnormal_products[
    abnormal_products["num_teams"] > 1
]
print(abnormal_products)
    

Empty DataFrame
Columns: [product_id, num_teams]
Index: []


In [28]:
# Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.
# datetime conversion
payment_product['report_month'] = pd.to_datetime(payment_product['report_month'])

# filter from Q2 2023
filtered = payment_product[
    (payment_product['report_month'] >= '2023-04-01') &
    (payment_product['team_own'].notna())
]

# lowest performance team
lowest_team = (
    filtered
    .groupby('team_own')['volume']
    .sum()
    .idxmin()
)

# least contributing category
lowest_category = (
    filtered[filtered['team_own'] == lowest_team]
    .dropna(subset=['category'])
    .groupby('category')['volume']
    .sum()
    .idxmin()
)

print("Lowest performance team:", lowest_team)
print("Lowest contributing category:", lowest_category)

 

Lowest performance team: APS
Lowest contributing category: PXXXXXE


In [29]:
# Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?

# filter refund transactions
refund_df = payment_product[payment_product['payment_group'] == 'refund']

# total refund volume by source
source_contribution = (
    refund_df
    .groupby('source_id')['volume']
    .sum()
    .reset_index()
)

# calculate contribution %
total_refund_volume = source_contribution['volume'].sum()

source_contribution['contribution_pct'] = (
    source_contribution['volume']
    / total_refund_volume
    * 100
)

# sort descending
source_contribution = source_contribution.sort_values(
    by='contribution_pct',
    ascending=False
)

print(source_contribution)

   source_id       volume  contribution_pct
1         38  36527454759         59.108225
2         39  16119059662         26.083641
0         37   9151069226         14.808134


In [36]:
conditions = [

    # transType = 2
    (df_transactions['transType'] == 2) & (df_transactions['merchant_id'] == 1205),

    (df_transactions['transType'] == 2) & (df_transactions['merchant_id'] == 2260),

    (df_transactions['transType'] == 2) & (df_transactions['merchant_id'] == 2270),

    (df_transactions['transType'] == 2),

    # transType = 8
    (df_transactions['transType'] == 8) & (df_transactions['merchant_id'] == 2250),

    (df_transactions['transType'] == 8)
]

choices = [

    'Bank Transfer Transaction',

    'Withdraw Money Transaction',

    'Top Up Money Transaction',

    'Payment Transaction',

    'Transfer Money Transaction',

    'Split Bill Transaction'
]

df_transactions['transaction_type'] = np.select(
    conditions,
    choices,
    default='Invalid Transaction'
)

print(df_transactions.head())

   transaction_id  merchant_id   volume  transType  transStatus  sender_id  \
0      3002692434            5   100000         24            1   10199794   
1      3002692437          305    20000          2            1   14022211   
2      3001960110         7255    48605         22            1       <NA>   
3      3002680710         2270  1500000          2            1   10059206   
4      3002680713         2275    90000          2            1   10004711   

   receiver_id extra_info      timeStamp          transaction_type  
0       199794        NaN  1682932054455       Invalid Transaction  
1     14022211        NaN  1682932054912       Payment Transaction  
2     10530940        NaN  1682932055000       Invalid Transaction  
3        59206        NaN  1682932055622  Top Up Money Transaction  
4         4711        NaN  1682932056197       Payment Transaction  


In [38]:
# exclude invalid transactions
valid_df = df_transactions[
    df_transactions['transaction_type'] != 'Invalid Transaction'
]

# aggregation
summary = (
    valid_df
    .groupby('transaction_type')
    .agg(
        number_of_transactions=('transaction_id', 'count'),
        total_volume=('volume', 'sum'),
        number_of_senders=('sender_id', 'nunique'),
        number_of_receivers=('receiver_id', 'nunique')
    )
    .reset_index()
)

print(summary)

             transaction_type  number_of_transactions  total_volume  \
0   Bank Transfer Transaction                   37879   50605806190   
1         Payment Transaction                  398665   71850608441   
2      Split Bill Transaction                    1376       4901464   
3    Top Up Money Transaction                  290498  108605618829   
4  Transfer Money Transaction                  341173   37032880492   
5  Withdraw Money Transaction                   33725   23418181420   

   number_of_senders  number_of_receivers  
0              23156                 9271  
1             139583               113298  
2               1323                  572  
3             110409               110409  
4              39021                34585  
5              24814                24814  
